In [1]:
import pandas as pd
import laspy
import numpy as np
import scipy
from scipy import spatial
import matplotlib
import matplotlib.pyplot as plt
import math
import open3d as o3d
from sklearn.cluster import DBSCAN
from collections import Counter
from sklearn.cluster import KMeans
import numpy as np
import random
from joblib import Parallel, delayed
#pcd1 = o3d.io.read_point_cloud("88-2xin_jiancai_1.ply")
#o3d.visualization.draw_geometries([pcd])
#from sklearn.decomposition import PCA
import numpy as np
import open3d as o3d
from scipy.linalg import svd, eig
from sklearn.neighbors import NearestNeighbors
from tqdm import tqdm

def fit_quadratic_surface(points, neighbors=30, verbose=True):

    if len(points) < neighbors:
        neighbors = len(points)
    
    nbrs = NearestNeighbors(n_neighbors=neighbors, algorithm='ball_tree').fit(points)
    distances, indices = nbrs.kneighbors(points)
    
    all_curvatures = []
    all_directions = []
    
    # 创建进度条
    if verbose:
        pbar = tqdm(total=len(points), 
                   desc="fit_quadratic_surface", 
                   unit="point",
                   ncols=100)
    
    success_count = 0
    fail_count = 0
    
    for i in range(len(points)):
        try:

            neighborhood = points[indices[i]]
            center_point = points[i]
   
            centered = neighborhood - center_point
            
            x = centered[:, 0]
            y = centered[:, 1]
            z = centered[:, 2]
      
            A = np.column_stack([
                x**2,           # a
                x * y,          # b  
                y**2,           # c
                x,              # d
                y,              # e
                np.ones(len(x)) # f
            ])
            
            try:
                U, s, Vt = svd(A, full_matrices=False)
                # 过滤小的奇异值以避免数值不稳定
                s_inv = np.zeros_like(s)
                mask = s > 1e-10
                s_inv[mask] = 1 / s[mask]
                coefficients = Vt.T @ (np.diag(s_inv) @ (U.T @ z))
            except:
    
                coefficients = np.linalg.pinv(A) @ z
            
            a, b, c, d, e, f = coefficients
        
            z_x = d
            z_y = e
            
            z_xx = 2 * a
            z_xy = b
            z_yy = 2 * c
            
            E = 1 + z_x**2
            F = z_x * z_y
            G = 1 + z_y**2
            
            denominator = np.sqrt(1 + z_x**2 + z_y**2)
            L = z_xx / denominator
            M = z_xy / denominator  
            N = z_yy / denominator
            
            shape_operator = np.array([
                [E, F],
                [F, G]
            ])
            
            second_fundamental = np.array([
                [L, M],
                [M, N]
            ])
            
            try:
                W = np.linalg.inv(shape_operator) @ second_fundamental
            except:
                W = np.linalg.pinv(shape_operator) @ second_fundamental
            
            eigenvalues, eigenvectors = eig(W)
            
            curvatures = np.real(eigenvalues)
            idx = np.argsort(np.abs(curvatures))[::-1]
            principal_curvatures = curvatures[idx]
            
            principal_directions = np.real(eigenvectors[:, idx])
            
            all_curvatures.append(principal_curvatures)
            all_directions.append(principal_directions)
            success_count += 1
            
        except Exception as e:

            all_curvatures.append(np.array([0.0, 0.0]))
            all_directions.append(np.eye(2))
            fail_count += 1
        
        # 更新进度条
        if verbose:
            pbar.update(1)

            if (i + 1) % 100 == 0:
                pbar.set_postfix({
                    'success': success_count,
                    'fail': fail_count,
                    'success_rate': f'{success_count/(i+1)*100:.1f}%'
                })
    
    if verbose:
        pbar.close()
        print(f"success {success_count}/{len(points)} point "
              f"({success_count/len(points)*100:.1f}%)")
        if fail_count > 0:
            print(f" {fail_count} pointsfail")
    
    return np.array(all_curvatures), np.array(all_directions)
def normalize_curvature_for_coloring(curvature_values):
  
    lower_bound = np.percentile(curvature_values, 5)
    upper_bound = np.percentile(curvature_values, 95)
    
    clipped = np.clip(curvature_values, lower_bound, upper_bound)
    
    normalized = (clipped - np.min(clipped)) / (np.max(clipped) - np.min(clipped))
    
    return normalized

def curvature_to_color(normalized_curvature, colormap_type='jet'):

    if colormap_type == 'jet':
        # Jet colormap
        colors = np.zeros((len(normalized_curvature), 3))
     
        for i, val in enumerate(normalized_curvature):
            if val < 0.125:
   
                r = 0
                g = 0
                b = 0.5 + 0.5 * (val / 0.125)
            elif val < 0.375:
               
                r = 0
                g = (val - 0.125) / 0.25
                b = 1
            elif val < 0.625:
      
                r = (val - 0.375) / 0.25
                g = 1
                b = 1 - (val - 0.375) / 0.25
            elif val < 0.875:
        
                r = 1
                g = 1 - (val - 0.625) / 0.25
                b = 0
            else:
          
                r = 1 - 0.5 * ((val - 0.875) / 0.125)
                g = 0
                b = 0
                
            colors[i] = [r, g, b]
            
    elif colormap_type == 'hot':
        # Hot colormap 
        colors = np.zeros((len(normalized_curvature), 3))
        
        for i, val in enumerate(normalized_curvature):
            if val < 0.333:
                r = val * 3
                g = 0
                b = 0
            elif val < 0.667:
                r = 1
                g = (val - 0.333) * 3
                b = 0
            else:
                r = 1
                g = 1
                b = (val - 0.667) * 3
                
            colors[i] = [r, g, b]
    
    elif colormap_type == 'coolwarm':
        # Cool-warm colormap 
        colors = np.zeros((len(normalized_curvature), 3))
        
        for i, val in enumerate(normalized_curvature):
            if val < 0.5:
             
                r = val * 2
                g = val * 2
                b = 1
            else:
            
                r = 1
                g = 2 - val * 2
                b = 2 - val * 2
                
            colors[i] = [r, g, b]
    
    return colors

def visualize_curvature_colored_point_cloud(points, principal_curvatures, curvature_type='k1', colormap='jet'):

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    
    if curvature_type == 'k1':
        curvature_values = principal_curvatures[:, 0]
        title_suffix = "first_curvatures (k1)"
    elif curvature_type == 'k2':
        curvature_values = principal_curvatures[:, 1]
        title_suffix = "gaussian_curvatures (k2)"
    elif curvature_type == 'gaussian':
        curvature_values = principal_curvatures[:, 0] * principal_curvatures[:, 1] 
        title_suffix = "second_curvatures"
    elif curvature_type == 'mean':
        curvature_values = (principal_curvatures[:, 0] + principal_curvatures[:, 1]) / 2
        title_suffix = "mean_curvatures"
    elif curvature_type == 'shape_index':
        # S = (2/π) * arctan((k1+k2)/(k1-k2))
        k1 = principal_curvatures[:, 0]
        k2 = principal_curvatures[:, 1]
        with np.errstate(divide='ignore', invalid='ignore'):
            shape_index = (2/np.pi) * np.arctan((k1 + k2) / (k1 - k2))
            shape_index = np.nan_to_num(shape_index, nan=0.0)
        curvature_values = shape_index
        title_suffix = "shape_index"
    else:
        raise ValueError("None")
    
    normalized_curvature = normalize_curvature_for_coloring(curvature_values)
    
    colors = curvature_to_color(normalized_curvature, colormap)
    
    pcd.colors = o3d.utility.Vector3dVector(colors)
    
    print(f"{title_suffix}")
    print(f" [{np.min(curvature_values):.4f}, {np.max(curvature_values):.4f}]")
    print(f" {colormap}")
    
    vis = o3d.visualization.Visualizer()
    vis.create_window(window_name=f" {title_suffix}", width=1200, height=800)
    vis.add_geometry(pcd)
    
    render_option = vis.get_render_option()
    render_option.point_size = 3.0
    render_option.background_color = np.array([0.1, 0.1, 0.1])  
    
    view_control = vis.get_view_control()
    view_control.set_front([0, 0, -1])
    view_control.set_lookat([0, 0, 0])
    view_control.set_up([0, -1, 0])
    view_control.set_zoom(0.8)
    
    vis.run()
    vis.destroy_window()
    
    return pcd
def compute_gaussian_mean_curvature(principal_curvatures):

    k1 = principal_curvatures[:, 0]
    k2 = principal_curvatures[:, 1]
    
    gaussian_curvature = k1 * k2
    mean_curvature = (k1 + k2) / 2
    
    return gaussian_curvature, mean_curvature

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
import numpy as np
import open3d as o3d

def adjust_normal_direction(normal_vector, reference_vector=None):
    if reference_vector is None:
        reference_vector = np.array([0, 0, 1])
    reference_vector = reference_vector / np.linalg.norm(reference_vector)
    if normal_vector.ndim == 1:
        normal_vector = normal_vector / np.linalg.norm(normal_vector)
        dot_product = np.dot(normal_vector, reference_vector)
        adjusted_vector = -normal_vector if dot_product < 0 else normal_vector
    else:
        norms = np.linalg.norm(normal_vector, axis=1, keepdims=True)
        normal_vector_normalized = normal_vector / norms
        
        dot_products = np.dot(normal_vector_normalized, reference_vector)
        adjusted_vector = normal_vector.copy()
        mask = dot_products < 0
        adjusted_vector[mask] = -normal_vector[mask]
    return adjusted_vector
def arcsin_and_arccos(pt):
    delta_y=pt[1]
    delta_x=pt[0]
    if delta_y==0 and delta_x==0:
        return 0
    sin = delta_x/(math.sqrt(delta_x**2 + delta_y**2))
    cos = delta_y/(math.sqrt(delta_x**2 + delta_y**2))
    if sin>=0 and cos>=0:
        return (math.asin(sin))/math.pi*180
    elif sin>=0 and cos<0:
        return (math.pi-math.asin(sin))/math.pi*180
    elif sin<0 and cos<0:
        return (math.pi-math.asin(sin))/math.pi*180
    elif sin<0 and cos>=0:
        return (2*math.pi+math.asin(sin))/math.pi*180

def arctans(pt):
    delta_z=pt[2]
    delta_y=pt[1]
    delta_x=pt[0]
    if delta_y==0 and delta_x==0 and delta_z!=0:
        return 0
    else:
        tan_1=np.abs((math.sqrt(delta_x**2 + delta_y**2)/(delta_z+0.000001)))
        return math.atan(tan_1)/math.pi*180
def apply_dbscan_to_cluster(points, eps, min_samples):
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    dbscan_labels = dbscan.fit_predict(points)
    n_dbscan_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
    n_noise = np.sum(dbscan_labels == -1)
    return dbscan_labels

def visualize_dbscan_results(original_points, dbscan_labels):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(original_points)
    n_clusters_dbscan = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
    colors = np.zeros((len(original_points), 3))
    for cluster_id in range(n_clusters_dbscan):
        mask = (dbscan_labels == cluster_id)
        color = plt.cm.tab20(cluster_id % 20)[:3] 
        colors[mask] = color
    noise_mask = (dbscan_labels == -1)
    colors[noise_mask] = [0.5, 0.5, 0.5] 
    pcd.colors = o3d.utility.Vector3dVector(colors)
    return pcd
def create_unified_visualization(lasdata, kmeans_labels, eps=0.12, min_samples=5):
    unified_pcd = o3d.geometry.PointCloud()
    unified_pcd.points = o3d.utility.Vector3dVector(lasdata)
    colors = np.zeros((len(lasdata), 3))

    global_dbscan_id = 0
    n_clusters = len(np.unique(kmeans_labels))
    
    for kmeans_id in range(n_clusters):
        cluster_mask = (kmeans_labels == kmeans_id)
        cluster_points = lasdata[cluster_mask]
        
        # Skip empty clusters
        if len(cluster_points) == 0:
            print(f"KMeans cluster {kmeans_id} is empty, skipping...")
            continue
            
        cluster_normals = nls[cluster_mask]
        
        # Handle empty normals array
        if len(cluster_normals) > 0:
            avg_normal = np.mean(cluster_normals, axis=0)
            avg_normal_normalized = avg_normal / np.linalg.norm(avg_normal)
        else:
            avg_normal_normalized = np.array([0, 0, 1])  # Default normal

        try:
            dbscan_labels = apply_dbscan_to_cluster(cluster_points, eps=eps, min_samples=min_samples)
            
            n_dbscan_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
            n_noise = np.sum(dbscan_labels == -1)
            
            for dbscan_id in range(n_dbscan_clusters):
                dbscan_mask = (dbscan_labels == dbscan_id)
                global_indices = np.where(cluster_mask)[0][dbscan_mask]
                color = plt.cm.tab20(global_dbscan_id % 20)[:3]
                colors[global_indices] = color
                global_dbscan_id += 1
            
            # Handle noise points
            noise_mask = (dbscan_labels == -1)
            if np.any(noise_mask):
                noise_indices = np.where(cluster_mask)[0][noise_mask]
                colors[noise_indices] = [0.3, 0.3, 0.3]
                
        except ValueError as e:
            print(f"Error processing KMeans cluster {kmeans_id}: {e}")
            # Assign a default color to this entire cluster
            global_indices = np.where(cluster_mask)[0]
            colors[global_indices] = [0.7, 0.7, 0.7]  # Light gray for failed clusters

    unified_pcd.colors = o3d.utility.Vector3dVector(colors)
    return unified_pcd


In [3]:
lasfile = r"XX.las"
inFile = laspy.read(lasfile)
x0,y0,z0 = inFile.x,inFile.y,inFile.z
lasdata = np.vstack((x0,y0,z0)).T

In [4]:
nls = np.loadtxt(r"XXX.txt", dtype=float, delimiter=' ')
nls = nls[:,-3:]
import math
import numpy as np
from sklearn.neighbors import KDTree
tree = spatial.cKDTree(lasdata) 
dist,nn_glob = tree.query(lasdata, k=30,workers=-1) 

In [5]:
nls = adjust_normal_direction(nls)

# KMEANS+DBSCAN

In [3]:
import numpy as np
import laspy
from scipy import spatial
from sklearn.cluster import KMeans
from sklearn.neighbors import KDTree
import time
start = time.time()
tree = spatial.cKDTree(lasdata)
dist, nn_glob = tree.query(lasdata, k=30, workers=-1)
n_clusters = 5 
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
labels = kmeans.fit_predict(nls)
cluster_centers = kmeans.cluster_centers_
print(f": {n_clusters}")
print(f": {np.bincount(labels)}")

for cluster_id in range(n_clusters):
    cluster_normals = nls[labels == cluster_id]
    avg_normal = np.mean(cluster_normals, axis=0)
    avg_normal_normalized = avg_normal / np.linalg.norm(avg_normal)
    print(f"{cluster_id}:")
    print(f"{len(cluster_normals)}")
    print(f"[{arcsin_and_arccos(avg_normal_normalized)}, {arctans(avg_normal_normalized)}]")

In [8]:
colors = plt.cm.tab10(labels % 10) 
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(lasdata[:, :3])
pcd.colors = o3d.utility.Vector3dVector(colors[:, :3])  
o3d.visualization.draw_geometries([pcd])

In [4]:
unified_pcd = create_unified_visualization(lasdata, labels, eps=0.12, min_samples=5)
o3d.visualization.draw_geometries([unified_pcd], 
                                 window_name="KMeans_DBSCAN",
                                 width=1200, 
                                 height=800)
total_time = time.time() - start
print(f" {total_time:.2f} ")

In [199]:
manual_colors = {
    0: [33,106,212],   
    1: [113,184,158],   
    2: [254,234,114],   
    3: [246,136,239], 
    4: [184,71,102]    
}

colors = np.zeros((len(labels), 3))
for cluster_id in range(n_clusters):
    mask = labels == cluster_id
    rgb_color = manual_colors.get(cluster_id, [128, 128, 128])  

    normalized_color = [c / 255.0 for c in rgb_color]
    colors[mask] = normalized_color

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(lasdata[:, :3])
pcd.colors = o3d.utility.Vector3dVector(colors)

o3d.visualization.draw_geometries([pcd])

In [207]:
header = laspy.LasHeader(point_format=3, version="1.3") 
header.offsets = np.min(lasdata, axis=0)

header.scales = [0.001, 0.001, 0.001]
las = laspy.LasData(header)

las.x = lasdata[:, 0]
las.y = lasdata[:, 1]
las.z = lasdata[:, 2]

las.red = colors[:, 0] * 65535  
las.green = colors[:, 1] * 65535
las.blue = colors[:, 2] * 65535

output_file = r"XXX.las"
las.write(output_file)

In [6]:
unified_pcd = o3d.geometry.PointCloud()
unified_pcd.points = o3d.utility.Vector3dVector(lasdata)
colors = np.zeros((len(lasdata), 3))
kmeans_labels = labels
global_dbscan_id = 0
n_clusters = len(np.unique(kmeans_labels))

for kmeans_id in range(n_clusters):
    cluster_mask = (kmeans_labels == kmeans_id)
    cluster_points = lasdata[cluster_mask]
    
    # Skip empty clusters
    if len(cluster_points) == 0:
        print(f"KMeans cluster {kmeans_id} is empty, skipping...")
        continue
        
    cluster_normals = nls[cluster_mask]
    
    # Handle empty normals array
    if len(cluster_normals) > 0:
        avg_normal = np.mean(cluster_normals, axis=0)
        avg_normal_normalized = avg_normal / np.linalg.norm(avg_normal)
    else:
        avg_normal_normalized = np.array([0, 0, 1])  # Default normal

    try:
        dbscan_labels = apply_dbscan_to_cluster(cluster_points, eps=0.12, min_samples=5)
        
        n_dbscan_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
        n_noise = np.sum(dbscan_labels == -1)
        

        for dbscan_id in range(n_dbscan_clusters):
            dbscan_mask = (dbscan_labels == dbscan_id)
            global_indices = np.where(cluster_mask)[0][dbscan_mask]
            color = plt.cm.tab20(global_dbscan_id % 20)[:3]
            colors[global_indices] = color
            global_dbscan_id += 1
        
        # Handle noise points
        noise_mask = (dbscan_labels == -1)
        if np.any(noise_mask):
            noise_indices = np.where(cluster_mask)[0][noise_mask]
            colors[noise_indices] = [0.3, 0.3, 0.3]
            
    except ValueError as e:
        print(f"Error processing KMeans cluster {kmeans_id}: {e}")
        # Assign a default color to this entire cluster
        global_indices = np.where(cluster_mask)[0]
        colors[global_indices] = [0.7, 0.7, 0.7]  # Light gray for failed clusters

unified_pcd.colors = o3d.utility.Vector3dVector(colors)

In [206]:
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(lasdata[:, :3])
pcd.colors = o3d.utility.Vector3dVector(colors)

# 显示结果
o3d.visualization.draw_geometries([pcd])

In [18]:
target_cluster = 1
cluster_points = lasdata[labels == target_cluster]

In [19]:
len(cluster_points)

514973

In [38]:
dbscan_labels = apply_dbscan_to_cluster(cluster_points, eps=0.12, min_samples=5)

In [7]:
dbscan_pcd = visualize_dbscan_results(cluster_points, dbscan_labels)
o3d.visualization.draw_geometries([dbscan_pcd], 
                                 window_name=f" {target_cluster} ")

# Hierarchical clustering+DBSCAN

In [11]:
import numpy as np
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score
import seaborn as sns
from sklearn.neighbors import NearestNeighbors

def hierarchical_clustering_normals(normals, method='ward', metric='euclidean', n_clusters=5):
    distance_matrix = pdist(normals, metric=metric)
    Z = linkage(distance_matrix, method=method)
    labels = fcluster(Z, n_clusters, criterion='maxclust')
    # Convert labels to start from 0
    labels = labels - 1
    return labels, Z

In [12]:
def sampled_hierarchical_clustering_with_analysis(normals, n_clusters=5, sample_size=10000, method='ward'):
    if len(normals) > sample_size:
        indices = np.random.choice(len(normals), sample_size, replace=False)
        sampled_normals = normals[indices]
        sampled_indices = indices
    else:
        sampled_normals = normals
        sampled_indices = np.arange(len(normals))

    distance_matrix = pdist(sampled_normals, metric='euclidean')

    Z = linkage(distance_matrix, method=method)
    
    sampled_labels = fcluster(Z, n_clusters, criterion='maxclust')
    sampled_labels = sampled_labels - 1  
    
    knn = NearestNeighbors(n_neighbors=1)
    knn.fit(sampled_normals)
    _, nearest_indices = knn.kneighbors(normals)

    all_labels = sampled_labels[nearest_indices.flatten()]
    
    return all_labels, Z, sampled_normals, sampled_labels

def calculate_cluster_statistics(normals, labels):
    unique_labels = np.unique(labels)
    cluster_stats = {}

    for label in unique_labels:
        cluster_mask = (labels == label)
        cluster_normals = normals[cluster_mask]
        cluster_size = len(cluster_normals)
        
        mean_normal = np.mean(cluster_normals, axis=0)
        
        mean_normal_length = np.linalg.norm(mean_normal)
        
        std_normal = np.std(cluster_normals, axis=0)
        
        angles_deg = np.degrees(np.arccos(np.abs(mean_normal / mean_normal_length)))
        
        cluster_stats[label] = {
            'size': cluster_size,
            'mean_normal': mean_normal
        }
    
    return cluster_stats

In [15]:
start = time.time()
n_clusters = 5
sample_size = 10000 

labels, linkage_matrix, sampled_normals, sampled_labels = sampled_hierarchical_clustering_with_analysis(
    nls, n_clusters=n_clusters, sample_size=sample_size, method='ward'
)
cluster_stats = calculate_cluster_statistics(nls, labels)

In [16]:
for cluster_id in range(n_clusters):
    cluster_normals = nls[labels == cluster_id]
    avg_normal = np.mean(cluster_normals, axis=0)
    avg_normal_normalized = avg_normal / np.linalg.norm(avg_normal)
    print(f" {cluster_id}:")
    print(f" {len(cluster_normals)}points")
    print(f" [{arcsin_and_arccos(avg_normal_normalized)}, {arctans(avg_normal_normalized)}]")

In [213]:
manual_colors = {
    0: [246,136,239],  
    1: [33,106,212],   
    2: [113,184,158],
    3: [184,71,102], 
    4: [254,234,114]    
}

colors = np.zeros((len(labels), 3))
for cluster_id in range(n_clusters):
    mask = labels == cluster_id
    rgb_color = manual_colors.get(cluster_id, [128, 128, 128])

    normalized_color = [c / 255.0 for c in rgb_color]
    colors[mask] = normalized_color

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(lasdata[:, :3])
pcd.colors = o3d.utility.Vector3dVector(colors)

o3d.visualization.draw_geometries([pcd])

In [217]:
header = laspy.LasHeader(point_format=3, version="1.3")
header.offsets = np.min(lasdata, axis=0)

header.scales = [0.001, 0.001, 0.001]
las = laspy.LasData(header)

las.x = lasdata[:, 0]
las.y = lasdata[:, 1]
las.z = lasdata[:, 2]

las.red = colors[:, 0] * 65535
las.green = colors[:, 1] * 65535
las.blue = colors[:, 2] * 65535

output_file = r"XXX.las"
las.write(output_file)

In [17]:
unified_pcd = o3d.geometry.PointCloud()
unified_pcd.points = o3d.utility.Vector3dVector(lasdata)
colors = np.zeros((len(lasdata), 3))
kmeans_labels = labels
global_dbscan_id = 0
n_clusters = len(np.unique(kmeans_labels))

for kmeans_id in range(n_clusters):
    cluster_mask = (kmeans_labels == kmeans_id)
    cluster_points = lasdata[cluster_mask]
    
    # Skip empty clusters
    if len(cluster_points) == 0:
        print(f"KMeans cluster {kmeans_id} is empty, skipping...")
        continue
        
    cluster_normals = nls[cluster_mask]
    
    # Handle empty normals array
    if len(cluster_normals) > 0:
        avg_normal = np.mean(cluster_normals, axis=0)
        avg_normal_normalized = avg_normal / np.linalg.norm(avg_normal)
    else:
        avg_normal_normalized = np.array([0, 0, 1])  # Default normal

    try:
        dbscan_labels = apply_dbscan_to_cluster(cluster_points, eps=0.12, min_samples=5)
        
        n_dbscan_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
        n_noise = np.sum(dbscan_labels == -1)
        
        for dbscan_id in range(n_dbscan_clusters):
            dbscan_mask = (dbscan_labels == dbscan_id)
            global_indices = np.where(cluster_mask)[0][dbscan_mask]
            color = plt.cm.tab20(global_dbscan_id % 20)[:3]
            colors[global_indices] = color
            global_dbscan_id += 1
        
        # Handle noise points
        noise_mask = (dbscan_labels == -1)
        if np.any(noise_mask):
            noise_indices = np.where(cluster_mask)[0][noise_mask]
            colors[noise_indices] = [0.3, 0.3, 0.3]
            
    except ValueError as e:
        print(f"Error processing KMeans cluster {kmeans_id}: {e}")
        # Assign a default color to this entire cluster
        global_indices = np.where(cluster_mask)[0]
        colors[global_indices] = [0.7, 0.7, 0.7]  # Light gray for failed clusters

unified_pcd.colors = o3d.utility.Vector3dVector(colors)

In [216]:
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(lasdata[:, :3])
pcd.colors = o3d.utility.Vector3dVector(colors)

o3d.visualization.draw_geometries([pcd])

In [40]:
target_cluster = 1
cluster_points = lasdata[labels == target_cluster]

In [104]:
dbscan_labels = apply_dbscan_to_cluster(cluster_points, eps=0.05, min_samples=3)

In [19]:
dbscan_pcd = visualize_dbscan_results(cluster_points, dbscan_labels)
o3d.visualization.draw_geometries([dbscan_pcd], 
                                 window_name=f"{target_cluster} DBSCAN")

In [18]:
unified_pcd = create_unified_visualization(lasdata, labels, eps=0.1, min_samples=4)
o3d.visualization.draw_geometries([unified_pcd], 
                                 window_name="DBSCAN",
                                 width=1200, 
                                 height=800)
total_time = time.time() - start
print(f"{total_time:.2f} ")

# RANSAC(cloudcompare)+Spectral clustering

In [18]:
import pickle
save_path = r"XXX.pkl"
with open(save_path, "rb") as f:
    planes = pickle.load(f)

In [19]:
point_cloud_objects = []
all_points = []
all_colors = []

for points in planes:
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    
    random_color = [random.random(), random.random(), random.random()]
    color_array = np.tile(random_color, (len(points), 1))
    
    pcd.colors = o3d.utility.Vector3dVector(color_array)
    point_cloud_objects.append(pcd)
    
    all_points.append(points)
    all_colors.append(color_array)

# 可视化
o3d.visualization.draw_geometries(point_cloud_objects)

In [1]:
import random
import numpy as np
import open3d as o3d
import laspy
import os
output_dir = "RANSAC+Spectral clustering"
os.makedirs(output_dir, exist_ok=True)

combined_points = np.vstack(all_points)
combined_colors = np.vstack(all_colors)

header = laspy.LasHeader(point_format=3, version="1.3")
header.scales = [0.001, 0.001, 0.001]

las = laspy.LasData(header)
las.x = combined_points[:, 0]
las.y = combined_points[:, 1]
las.z = combined_points[:, 2]

las.red = (combined_colors[:, 0] * 65535).astype(np.uint16)
las.green = (combined_colors[:, 1] * 65535).astype(np.uint16)
las.blue = (combined_colors[:, 2] * 65535).astype(np.uint16)

las_output_file = os.path.join(output_dir, "RANSAC+Spectral clustering.las")
las.write(las_output_file)
print(f": {las_output_file}")

In [26]:
header = laspy.LasHeader(point_format=3, version="1.3") 
header.scales = [0.001, 0.001, 0.001]
las = laspy.LasData(header)

all_points = []
all_colors = []

for i, (points, cluster_id) in enumerate(zip(planes, cluster_labels)):
    all_points.append(points)
    
    if cluster_id in manual_colors:
        color_rgb = manual_colors[cluster_id]
        normalized_color = [c / 255.0 for c in color_rgb]
    else:
        normalized_color = [0.5, 0.5, 0.5]  
    
    color_array = np.tile(normalized_color, (len(points), 1))
    all_colors.append(color_array)

combined_points = np.vstack(all_points)
combined_colors = np.vstack(all_colors)

las.x = combined_points[:, 0]
las.y = combined_points[:, 1]
las.z = combined_points[:, 2]

las.red = (combined_colors[:, 0] * 65535).astype(np.uint16)
las.green = (combined_colors[:, 1] * 65535).astype(np.uint16)
las.blue = (combined_colors[:, 2] * 65535).astype(np.uint16)

output_file = r"XXX.las"
las.write(output_file)

In [17]:
import numpy as np
import open3d as o3d
from sklearn.decomposition import PCA
from sklearn.cluster import SpectralClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
def compute_normals_with_pca(points):
    pca = PCA(n_components=3)
    pca.fit(points)
    
    normal_vector = pca.components_[2]
    
    if normal_vector[2] < 0:
        normal_vector = -normal_vector
    
    return normal_vector

def spectral_cluster_normals(normals, n_clusters=2):

    similarity_matrix = cosine_similarity(normals)
    
    similarity_matrix = (similarity_matrix + 1) / 2
    
    spectral = SpectralClustering(
        n_clusters=n_clusters,
        affinity='precomputed',
        random_state=42
    )
    
    labels = spectral.fit_predict(similarity_matrix)
    return labels

In [27]:
start = time.time()

normals_list = []
for i, points in enumerate(planes):
    normal = compute_normals_with_pca(points)
    normals_list.append(normal)


In [3]:
normals_array = np.array(normals_list)
n_clusters = 5
cluster_labels = spectral_cluster_normals(normals_array, n_clusters=n_clusters)

In [2]:
manual_colors = {
    0: [184, 71, 102],
    1: [113, 184, 158],   
    2: [246, 136, 239],  
    3: [254, 234, 114],   
    4: [33, 106, 212]   
}

normalized_colors = {}
for cluster_id, color in manual_colors.items():
    normalized_colors[cluster_id] = [c / 255.0 for c in color]

point_cloud_objects = []

for i, (points, cluster_id) in enumerate(zip(planes, cluster_labels)):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    
    if cluster_id in normalized_colors:
        color = normalized_colors[cluster_id]
    else:
        color = [0.5, 0.5, 0.5]
    
    color_array = np.tile(color, (len(points), 1))
    pcd.colors = o3d.utility.Vector3dVector(color_array)
    
    point_cloud_objects.append(pcd)

for cluster_id, color in manual_colors.items():
    normalized_color = normalized_colors[cluster_id]
    print(f"{cluster_id}: RGB({color[0]}, {color[1]}, {color[2]}) -> Normalized({normalized_color[0]:.3f}, {normalized_color[1]:.3f}, {normalized_color[2]:.3f})")

o3d.visualization.draw_geometries(point_cloud_objects) 

total_time = time.time() - start
print(f" {total_time:.2f} ")

In [4]:
clustered_planes = {}
clustered_normals = {}

for i, (points, normal, label) in enumerate(zip(planes, normals_array, cluster_labels)):
    if label not in clustered_planes:
        clustered_planes[label] = []
        clustered_normals[label] = []
    
    clustered_planes[label].append(points)
    clustered_normals[label].append(normal)

for cluster_id in sorted(clustered_planes.keys()):
    print(f" {cluster_id}: {len(clustered_planes[cluster_id])} points")

average_normals = {}

for cluster_id, normals in clustered_normals.items():
    cluster_normals_array = np.array(normals)
    
    avg_normal = np.mean(cluster_normals_array, axis=0)
    avg_normal = avg_normal / np.linalg.norm(avg_normal)
    
    average_normals[cluster_id] = avg_normal
    
    print(f"[{arcsin_and_arccos(avg_normal)}, {arctans(avg_normal)}]")

total_time = time.time() - start  
print(f": {total_time:.2f} ")

# Eigenvalue ratio+CFSFDP

In [61]:
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA

def compute_point_features(points, k_neighbors=30):
    n_points = points.shape[0]
    eigenvalues = np.zeros((n_points, 3))
    lambda_ratios = np.zeros(n_points)
    
    nbrs = NearestNeighbors(n_neighbors=k_neighbors, algorithm='ball_tree').fit(points)
    distances, indices = nbrs.kneighbors(points)
    
    for i in range(n_points):

        neighborhood_indices = indices[i]
        neighborhood_points = points[neighborhood_indices]
        
        centroid = np.mean(neighborhood_points, axis=0)
        
        centered_points = neighborhood_points - centroid
        covariance_matrix = np.dot(centered_points.T, centered_points) / (k_neighbors - 1)
        
        eigvals, eigvecs = np.linalg.eigh(covariance_matrix)
        
        eigvals_sorted = np.sort(eigvals)[::-1]
        eigenvalues[i] = eigvals_sorted
        
        lambda_sum = np.sum(eigvals_sorted)
        if lambda_sum > 1e-8:
            lambda_ratios[i] = eigvals_sorted[0] / lambda_sum
        else:
            lambda_ratios[i] = 0.0
    
    return lambda_ratios, eigenvalues

def split_points_by_ratio(points, lambda_ratios, threshold):

    less_mask = lambda_ratios < threshold
    greater_mask = lambda_ratios >= threshold
    
    less_indices = np.where(less_mask)[0]
    greater_indices = np.where(greater_mask)[0]
    
    less_points = points[less_indices]
    greater_points = points[greater_indices]
    
    less_ratios = lambda_ratios[less_indices]
    greater_ratios = lambda_ratios[greater_indices]
    
    result = {
        'less_than_threshold': {
            'points': less_points,
            'indices': less_indices,
            'ratios': less_ratios,
            'count': len(less_points)
        },
        'greater_than_threshold': {
            'points': greater_points,
            'indices': greater_indices,
            'ratios': greater_ratios,
            'count': len(greater_points)
        },
        'threshold': threshold,
        'total_points': len(points)
    }
    
    return result

def visualize_split_points(original_points, split_result):

    less_points = split_result['less_than_threshold']['points']
    greater_points = split_result['greater_than_threshold']['points']
    threshold = split_result['threshold']
    
    less_pcd = o3d.geometry.PointCloud()
    less_pcd.points = o3d.utility.Vector3dVector(less_points)
    less_pcd.paint_uniform_color([1, 0, 0]) 
    
    greater_pcd = o3d.geometry.PointCloud()
    greater_pcd.points = o3d.utility.Vector3dVector(greater_points)
    greater_pcd.paint_uniform_color([0, 0, 1])
    
    o3d.visualization.draw_geometries([less_pcd, greater_pcd])

In [62]:
lambda_ratios, eigenvalues = compute_point_features(lasdata, k_neighbors=30)
threshold = 0.5
split_result = split_points_by_ratio(lasdata, lambda_ratios, threshold)
visualize_split_points(lasdata, split_result)

In [74]:
start = time.time()

In [75]:
lambda_ratios = lambda_ratios.reshape(-1, 1)

In [76]:
cfs = split_result['greater_than_threshold']['points']

In [77]:
point_indices = find_point_indices(cfs, lasdata)
subset_normals = nls[point_indices]

In [78]:
qxqj = []
for i in range(len(subset_normals)):
    qxqj.append((arcsin_and_arccos(subset_normals[i]),arctans(subset_normals[i])))

In [34]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.neighbors import NearestNeighbors
from scipy.spatial.distance import cdist
import warnings
warnings.filterwarnings('ignore')

def cfsfdp_large_scale(X, n_neighbors=100, auto_select_centers=True, n_centers=None, plot_decision=False):

    n_samples = X.shape[0]
    print(f"{n_samples} ")
    

    nbrs = NearestNeighbors(n_neighbors=min(n_neighbors, n_samples-1))
    nbrs.fit(X)
    distances, indices = nbrs.kneighbors(X)
    
    dc = np.mean(distances[:, 1:])
    rho = np.zeros(n_samples)
    
    for i in tqdm(range(n_samples), desc="calculate density"):

        rho[i] = np.sum(np.exp(-(distances[i, 1:] / dc)**2))
    

    delta = np.zeros(n_samples)
    nearest_higher_density = np.zeros(n_samples, dtype=int)
    
    rho_sorted_indices = np.argsort(-rho)
    
    from sklearn.neighbors import KDTree
    tree = KDTree(X)
    
    for i, idx in enumerate(tqdm(rho_sorted_indices, desc="high density points")):
        if i == 0: 

            delta[idx] = np.max(cdist(X[idx:idx+1], X)[0])
            nearest_higher_density[idx] = idx
        else:

            higher_density_indices = rho_sorted_indices[:min(i, 1000)]
            
            if len(higher_density_indices) > 0:

                dists = cdist(X[idx:idx+1], X[higher_density_indices])[0]
                min_dist_idx = np.argmin(dists)
                delta[idx] = dists[min_dist_idx]
                nearest_higher_density[idx] = higher_density_indices[min_dist_idx]
            else:
                delta[idx] = np.max(cdist(X[idx:idx+1], X)[0])
                nearest_higher_density[idx] = idx
    
    if auto_select_centers:

        if n_centers is None:

            product = rho * delta

            sorted_product = np.sort(product)[::-1]
            diffs = np.diff(sorted_product)
            if len(diffs) > 10:

                n_centers = np.argmax(diffs[10:]) + 10 + 1
            else:
                n_centers = min(10, len(sorted_product))
        
        n_centers = min(n_centers, n_samples)
        center_indices = np.argsort(-rho * delta)[:n_centers]
    else:

        if plot_decision and n_samples <= 10000:
            plot_decision_graph(rho, delta)
        
        if n_centers is None:
            n_centers = int(input("clusters input: "))
        
        center_indices = np.argsort(-rho * delta)[:n_centers]
    
    labels = -np.ones(n_samples, dtype=int)
    
    for i, center_idx in enumerate(center_indices):
        labels[center_idx] = i
    
    unassigned_mask = labels == -1
    unassigned_indices = rho_sorted_indices[unassigned_mask[rho_sorted_indices]]
    
    for idx in tqdm(unassigned_indices, desc="pints to cluster"):
        if labels[idx] == -1:
            labels[idx] = labels[nearest_higher_density[idx]]
    
    return labels, rho, delta, center_indices

def plot_decision_graph(rho, delta, highlight_indices=None):

    plt.figure(figsize=(10, 6))
    plt.scatter(rho, delta, s=10, alpha=0.6)
    
    if highlight_indices is not None:
        plt.scatter(rho[highlight_indices], delta[highlight_indices], 
                   c='red', s=50, marker='*', label='Cluster Centers')
        plt.legend()
    
    plt.xlabel('Density (rho)')
    plt.ylabel('Distance (delta)')
    plt.title('Decision Graph')
    plt.grid(True, alpha=0.3)
    plt.show()

def cfsfdp_sampling(X, sample_ratio=0.1, **kwargs):

    n_samples = X.shape[0]
    sample_size = int(n_samples * sample_ratio)
    
    sample_indices = np.random.choice(n_samples, sample_size, replace=False)
    X_sampled = X[sample_indices]
    
    labels_sampled, rho_sampled, delta_sampled, centers_sampled = cfsfdp_large_scale(
        X_sampled, **kwargs
    )
    

    from sklearn.neighbors import NearestNeighbors
    nbrs = NearestNeighbors(n_neighbors=1)
    nbrs.fit(X_sampled)
    
    remaining_indices = np.setdiff1d(np.arange(n_samples), sample_indices)
    _, assigned_indices = nbrs.kneighbors(X[remaining_indices])
    
    full_labels = np.zeros(n_samples, dtype=int)
    full_labels[sample_indices] = labels_sampled
    full_labels[remaining_indices] = labels_sampled[assigned_indices.flatten()]
    
    return full_labels, labels_sampled, sample_indices

In [80]:
qxqj = np.array(qxqj)

In [23]:
full_labels, sampled_labels, sample_indices = cfsfdp_sampling(
    qxqj, 
    sample_ratio=0.001,
    n_neighbors=50,
    n_centers=5)

In [82]:
len(subset_normals)

996528

In [83]:
np.unique(full_labels)

array([0, 1, 2, 3, 4])

In [72]:
manual_colors = {
    0: [113,184,158],  
    1: [184,71,102],   
    2: [246,136,239],
    3: [254,234,114], 
    4: [33,106,212]   
}

cluster_colors = {}
for cluster_id in set(full_labels):
    if cluster_id == -1:
        cluster_colors[cluster_id] = [0.5, 0.5, 0.5]
    elif cluster_id in manual_colors:

        color_rgb = manual_colors[cluster_id]
        cluster_colors[cluster_id] = [c / 255.0 for c in color_rgb]
    else:

        cluster_colors[cluster_id] = [random.random(), random.random(), random.random()]

colors = []
for label in full_labels:
    colors.append(cluster_colors[label])

colors = np.array(colors)

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(cfs)
pcd.colors = o3d.utility.Vector3dVector(colors)

o3d.visualization.draw_geometries([pcd], window_name="result")

In [22]:
unique_labels = np.unique(full_labels)
stats = {}
for label in unique_labels:
    if label == -1:
        continue  
    mask = (full_labels == label)
    cluster_points = qxqj[mask]
    centroid = np.mean(cluster_points, axis=0)  
    print(centroid)

In [21]:
combined_pcd = simple_process_all_clusters_tab20(cfs, full_labels)

In [60]:
header = laspy.LasHeader(point_format=3, version="1.3")
header.offsets = np.min(sorted_arrays[0], axis=0)

header.scales = [0.001, 0.001, 0.001]
las = laspy.LasData(header)

las.x = cfs[:, 0]
las.y = cfs[:, 1]
las.z = cfs[:, 2]

las.red = colors[:, 0] * 65535
las.green = colors[:, 1] * 65535
las.blue = colors[:, 2] * 65535

output_file = r"XXX.las"
las.write(output_file)

In [20]:
unique_labels = np.unique(full_labels)
unique_labels = unique_labels[unique_labels != -1]

all_points = []
all_colors = []
all_cluster_info = []

color_counter = 0

for cluster_label in unique_labels:
    cluster_mask = (full_labels == cluster_label)
    cluster_points = cfs[cluster_mask]
    
    dbscan_labels = apply_dbscan_to_cluster(cluster_points, 0.06, 3)
    
    colors = np.zeros((len(cluster_points), 3))
    n_subclusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
    
    for sub_id in range(n_subclusters):
        mask = (dbscan_labels == sub_id)
        color = plt.cm.tab20(color_counter % 20)[:3]
        colors[mask] = color
        color_counter += 1
    
    noise_mask = (dbscan_labels == -1)
    colors[noise_mask] = [0.2, 0.2, 0.2]
    
    all_points.append(cluster_points)
    all_colors.append(colors)
    all_cluster_info.extend([f"Cluster_{cluster_label}_Sub_{dbscan_labels[i]}" for i in range(len(dbscan_labels))])
    
    print(f" {cluster_label}: {len(cluster_points)}points -> {n_subclusters}")

total_time = time.time() - start
print(f" {total_time:.2f}")

In [ ]:
if all_points:

    combined_points = np.vstack(all_points)
    combined_colors = np.vstack(all_colors)
    
    header = laspy.LasHeader(point_format=3, version="1.3")
    header.scales = [0.001, 0.001, 0.001]
    header.offsets = np.min(combined_points, axis=0)
    
    las = laspy.LasData(header)
    las.x = combined_points[:, 0]
    las.y = combined_points[:, 1]
    las.z = combined_points[:, 2]
    
    las.red = (combined_colors[:, 0] * 65535).astype(np.uint16)
    las.green = (combined_colors[:, 1] * 65535).astype(np.uint16)
    las.blue = (combined_colors[:, 2] * 65535).astype(np.uint16)
    
    output_path = r"XXX.las"
    las.write(output_path)

# Region growing + meanshift

In [86]:
def random_color():
    color_arr = ['1', '2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'C', 'D', 'E', 'F']
    color = ""
    for i in range(6):
        color += color_arr[random.randint(0, 14)]
    return "#" + color

def hex_to_rgb(hex_color):
    digit = list(map(str, range(10))) + list("ABCDEF")
    hex_color = hex_color.lstrip('#')
    return tuple(int(hex_color[i:i+2], 16) for i in (0, 2, 4))

def assign_color_to_point_cloud(point_cloud):
    hex_color = random_color()
    rgb_color = hex_to_rgb(hex_color)
    colored_point_cloud = []

    for point in point_cloud:
        colored_point_cloud.append((*point, *rgb_color))

    return colored_point_cloud

def normalsestimation(pointcloud):
    curvatures = np.empty((len(pointcloud),1))
    for index in range(len(pointcloud)):
        if index-(index//10000*10000)==0:
            print(index)
        nn_loc = pointcloud[nn_glob[index]]
        idx, _ = PCA(nn_loc)
        curvatures[index] = idx[-1]/ np.sum(idx)
    return curvatures

import numpy as np
from scipy.spatial import KDTree
import matplotlib.pyplot as plt

def compute_curvature_based_on_normal_difference(points, normals, k=30):
    num_points = len(points)

    kdtree = KDTree(points)

    curvatures = np.zeros(num_points)

    distances, indices = kdtree.query(points, k=k+1)

    for i in range(num_points):
        normal_i = normals[i]
        neighbor_indices = indices[i]

        neighbor_indices = neighbor_indices[neighbor_indices != i]

        if len(neighbor_indices) == 0:
            curvature = 0.0
        else:
            
            neighbor_normals = normals[neighbor_indices]

            dot_products = np.clip(np.dot(neighbor_normals, normal_i), -1.0, 1.0)

            angle_diffs = np.arccos(dot_products)

            curvature = np.mean(angle_diffs)

        curvatures[i] = curvature

    return curvatures
def regiongrowing1(pointcloud, curvatures, nn_glob, nls, theta_th, cur_th, neighborhood_size=30):
    order_set = set(np.argsort(curvatures[:, 0]))
    region = []
  
    theta_threshold = theta_th
    regions_found = set()

    region_index = 1

    while order_set:
        region_cur = []
        seed_cur = []
        poi_min = order_set.pop()
        region_cur.append(poi_min)
        seedval = 0
        seed_cur.append(poi_min)
        regions_found.add(poi_min)

        while seedval < len(seed_cur):
            nn_loc = nn_glob[seed_cur[seedval]]
            seed_normal = nls[seed_cur[seedval]]
            #seed_point = pointcloud[seed_cur[seedval]]

            seed_angle_sin_cos, seed_angle_tan = nls_angles[seed_cur[seedval]]

            for nn_cur in nn_loc:
                if nn_cur in regions_found:
                    continue

                nn_angle_sin_cos, nn_angle_tan = nls_angles[nn_cur]
                
                qingxiang_vect = np.abs(seed_angle_sin_cos - nn_angle_sin_cos)
                qingjiao_vect = np.abs(seed_angle_tan - nn_angle_tan)

                if np.all([qingxiang_vect < theta_threshold, qingjiao_vect < theta_threshold]):
                    region_cur.append(nn_cur)
                if nn_cur in order_set:
                    order_set.remove(nn_cur)
                    regions_found.add(nn_cur)
                    if curvatures[nn_cur] < cur_th:
                        seed_cur.append(nn_cur)
            seedval += 1

        region_index += 1

        region.append(region_cur)
    
    return region

def refine_regions_with_dynamic_kmeans_optimized(regions, nls, max_inertia_threshold, min_points, n_jobs=-1):
 
    def process_region(region_cur):
        if len(region_cur) < min_points:
            return [region_cur]
        region_normals = nls[region_cur]
        kmeans_1 = KMeans(n_clusters=1, init='k-means++', n_init=10, random_state=42)
        kmeans_1.fit(region_normals)
        inertia_1 = kmeans_1.inertia_  

        if inertia_1 > max_inertia_threshold:
            kmeans_2 = KMeans(n_clusters=2, init='k-means++', n_init=10, random_state=42)
            labels = kmeans_2.fit_predict(region_normals)
            subregions = [
                [region_cur[i] for i in range(len(region_cur)) if labels[i] == cluster_label]
                for cluster_label in range(2)
            ]
            return subregions
        else:
            return [region_cur]
    refined_regions = Parallel(n_jobs=n_jobs)(
        delayed(process_region)(region_cur) for region_cur in regions
    )
    return [subregion for region in refined_regions for subregion in region]

In [ ]:
curvatures = compute_curvature_based_on_normal_difference(lasdata, nls)

In [87]:
curvatures = np.array(curvatures)
curvatures = curvatures.reshape(-1,1)
len(curvatures)

1044090

In [88]:
arcsin_and_arccos_values = [arcsin_and_arccos(normal) for normal in nls if arcsin_and_arccos(normal) is not None]
arctans_values = [arctans(normal) for normal in nls if arctans(normal) is not None]

default_arcsin_and_arccos = np.median(arcsin_and_arccos_values) if arcsin_and_arccos_values else np.pi / 2
default_arctans = np.median(arctans_values) if arctans_values else 0.0

nls_angles = []
for i, normal in enumerate(nls):
    angle_sin_cos = arcsin_and_arccos(normal)
    angle_tan = arctans(normal)

    if angle_sin_cos is None or angle_tan is None:

        nn_loc = nn_glob[i][:30]  
        neighbor_angles_sin_cos = [arcsin_and_arccos(nls[j]) for j in nn_loc]
        neighbor_angles_tan = [arctans(nls[j]) for j in nn_loc]

        valid_sin_cos = [val for val in neighbor_angles_sin_cos if val is not None]
        valid_tan = [val for val in neighbor_angles_tan if val is not None]

        if valid_sin_cos and valid_tan:
            angle_sin_cos = np.mean(valid_sin_cos)
            angle_tan = np.mean(valid_tan)
        else:
            
            angle_sin_cos = default_arcsin_and_arccos
            angle_tan = default_arctans

    nls_angles.append((angle_sin_cos, angle_tan))

In [110]:
start = time.time()

In [111]:
region = regiongrowing1(lasdata, curvatures, nn_glob, nls, theta_th=12, cur_th=0.08,neighborhood_size=30)

In [112]:
refined_region = []
for i in range(len(region)):
    if len(region[i])>=3:
        refined_region.append(region[i])  

In [113]:
normals_list = []
for i, points in enumerate(refined_region):
    points = lasdata[points]
    normal = compute_normals_with_pca(points)
    normals_list.append(normal)


In [24]:
import numpy as np
from sklearn.cluster import MeanShift
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import estimate_bandwidth
import matplotlib.pyplot as plt

scaler = StandardScaler()
points_scaled = scaler.fit_transform(normals_list)

bandwidth = estimate_bandwidth(points_scaled, quantile=0.045)
print(f"Estimated bandwidth: {bandwidth}")

ms = MeanShift(bandwidth=bandwidth, bin_seeding=True)
ms.fit(points_scaled)

labels = ms.labels_
cluster_centers = ms.cluster_centers_

n_clusters = len(np.unique(labels))
print(f"Number of clusters: {n_clusters}")

In [115]:
normals_list = np.array(normals_list)

In [116]:
for i in range(n_clusters):

    cluster_points = points_scaled[labels == i]
    cluster_original_points = normals_list[labels == i]
    
    cluster_mean = np.mean(cluster_points, axis=0)
    
    cluster_mean_original = np.mean(cluster_original_points, axis=0)

In [108]:
point_cloud_objects = []
all_points = []
all_colors = []

for i in range(len(refined_region)):
    points = lasdata[refined_region[i]]
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    
    random_color = [random.random(), random.random(), random.random()]
    color_array = np.tile(random_color, (len(points), 1))
    
    pcd.colors = o3d.utility.Vector3dVector(color_array)
    point_cloud_objects.append(pcd)
    
    all_points.append(points)
    all_colors.append(color_array)

o3d.visualization.draw_geometries(point_cloud_objects)

In [5]:
import random
import numpy as np
import open3d as o3d
import laspy
import os
output_dir = ""
os.makedirs(output_dir, exist_ok=True)

combined_points = np.vstack(all_points)
combined_colors = np.vstack(all_colors)

header = laspy.LasHeader(point_format=3, version="1.3")
header.scales = [0.001, 0.001, 0.001]

las = laspy.LasData(header)
las.x = combined_points[:, 0]
las.y = combined_points[:, 1]
las.z = combined_points[:, 2]

las.red = (combined_colors[:, 0] * 65535).astype(np.uint16)
las.green = (combined_colors[:, 1] * 65535).astype(np.uint16)
las.blue = (combined_colors[:, 2] * 65535).astype(np.uint16)

las_output_file = os.path.join(output_dir, "XXX.las")
las.write(las_output_file)
print(f": {las_output_file}")

In [6]:
manual_colors = {
    0: [113, 184, 158],
    1: [254, 234, 114],   
    2: [184, 71, 102],
    3: [246, 136, 239],   
    4: [33, 106, 212]  
}

normalized_colors = {}
for cluster_id, color in manual_colors.items():
    normalized_colors[cluster_id] = [c / 255.0 for c in color]

point_cloud_objects = []

for i, (points, cluster_id) in enumerate(zip(refined_region, labels)):
    points = lasdata[refined_region[i]]
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    
    if cluster_id in normalized_colors:
        color = normalized_colors[cluster_id]
    else:
        color = [0.5, 0.5, 0.5]
    
    color_array = np.tile(color, (len(points), 1))
    pcd.colors = o3d.utility.Vector3dVector(color_array)
    
    point_cloud_objects.append(pcd)

for cluster_id, color in manual_colors.items():
    normalized_color = normalized_colors[cluster_id]
    print(f" {cluster_id}: RGB({color[0]}, {color[1]}, {color[2]}) -> Normalized({normalized_color[0]:.3f}, {normalized_color[1]:.3f}, {normalized_color[2]:.3f})")

o3d.visualization.draw_geometries(point_cloud_objects)
total_time = time.time() - start
print(f"{total_time:.2f}")

In [7]:
normalized_colors = {}
for cluster_id, color in manual_colors.items():
    normalized_colors[cluster_id] = [c / 255.0 for c in color]

point_cloud_objects = []
all_points = []
all_colors = []
all_labels = []

for i, (points, cluster_id) in enumerate(zip(refined_region, labels)):
    points = lasdata[refined_region[i]]
    
    if cluster_id in normalized_colors:
        color = normalized_colors[cluster_id]
    else:
        color = [0.5, 0.5, 0.5]
    
    color_array = np.tile(color, (len(points), 1))
    
    all_points.append(points)
    all_colors.append(color_array)
    all_labels.extend([cluster_id] * len(points))
    
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    pcd.colors = o3d.utility.Vector3dVector(color_array)
    point_cloud_objects.append(pcd)

all_points = np.vstack(all_points)
all_colors = np.vstack(all_colors)

def save_pointcloud_with_rgb_as_las(points, colors, labels, filename="clustered_pointcloud.las"):

    header = laspy.LasHeader(point_format=2, version="1.2")
    header.offsets = np.min(points, axis=0)
    header.scales = [0.01, 0.01, 0.01]
    
    las = laspy.LasData(header)
    
    las.x = points[:, 0]
    las.y = points[:, 1]
    las.z = points[:, 2]

    colors_16bit = (colors * 65535).astype(np.uint16)
    las.red = colors_16bit[:, 0]
    las.green = colors_16bit[:, 1]
    las.blue = colors_16bit[:, 2]
    
    las.user_data = np.array(labels, dtype=np.uint8)
    
    las.write(filename)

save_pointcloud_with_rgb_as_las(all_points, all_colors, all_labels, "XXX.las")

# Fuzzy C means+HDBSCAN

In [9]:
import numpy as np
import skfuzzy as fuzz
import colorsys
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
start = time.time()
data = nls
data = data.T

n_clusters = 5
m = 2.0  
max_iter = 100
error = 1e-5

cntr, u, u0, d, jm, p, fpc = fuzz.cluster.cmeans(
    data, n_clusters, m, error=error, maxiter=max_iter, init=None
)

In [10]:
labels = np.argmax(u, axis=0)

In [9]:
da = nls
for j in range(n_clusters):
    cluster_points = da[labels == j]
    actual_mean = np.mean(cluster_points, axis=0)
    n_points_cluster = len(cluster_points)
    print(f" {j}:")
    print(f" {n_points_cluster}")
    print(f" [{arcsin_and_arccos(actual_mean)},{arctans(actual_mean)}]")

In [12]:
manual_colors = {
    0: [246, 136, 239],  
    1: [113, 184, 158],   
    2: [254, 234, 114],  
    3: [33, 106, 212],   
    4: [184, 71, 102]  
}

colors = np.zeros((len(labels), 3))
for cluster_id in range(n_clusters):
    mask = labels == cluster_id
    rgb_color = manual_colors.get(cluster_id, [128, 128, 128])

    normalized_color = [c / 255.0 for c in rgb_color]
    colors[mask] = normalized_color

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(lasdata[:, :3])
pcd.colors = o3d.utility.Vector3dVector(colors)
o3d.visualization.draw_geometries([pcd])

In [96]:
header = laspy.LasHeader(point_format=3, version="1.3") 
header.offsets = np.min(lasdata, axis=0)

header.scales = [0.001, 0.001, 0.001]
las = laspy.LasData(header)

las.x = lasdata[:, 0]
las.y = lasdata[:, 1]
las.z = lasdata[:, 2]

las.red = colors[:, 0] * 65535
las.green = colors[:, 1] * 65535
las.blue = colors[:, 2] * 65535

output_file = r"XXX.las"
las.write(output_file)

In [13]:
import numpy as np
import hdbscan
import open3d as o3d
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [14]:
fcm_labels = np.argmax(u, axis=0)

fcm_colors = {
    0: [246, 136, 239],
    1: [33, 106, 212],
    2: [113, 184, 158],
    3: [184, 71, 102],
    4: [254, 234, 114]
}

def generate_color_variants(base_color, n_variants):

    base_rgb = np.array(base_color) / 255.0
    variants = []
    
    for i in range(n_variants):

        brightness_factor = 0.7 + 0.3 * (i / max(1, n_variants-1))
        variant = np.clip(base_rgb * brightness_factor, 0, 1)
        variants.append(variant)
    
    return variants

all_hdbscan_labels = np.full(len(fcm_labels), -1, dtype=int)
final_colors = np.zeros((len(fcm_labels), 3))
current_label_offset = 0

In [15]:
data = lasdata

In [10]:
def generate_distinct_colors(n_colors):
    colors = []
    for i in range(n_colors):
        hue = i / n_colors
        saturation = 0.7 + 0.3 * (i % 3) / 2  
        value = 0.8 + 0.2 * (i % 2)  
        rgb = colorsys.hsv_to_rgb(hue, saturation, value)
        colors.append(rgb)
    return colors
max_expected_subclusters = 1000
distinct_colors = generate_distinct_colors(max_expected_subclusters)

all_hdbscan_labels = np.full(len(fcm_labels), -1, dtype=int) 
final_colors = np.zeros((len(fcm_labels), 3))
subcluster_counter = 0 

In [11]:
if data.shape[0] == n_clusters:
    points_3d = data.T
else:
    points_3d = data

print(f"{points_3d.shape}")
print(f"{fcm_labels.shape}")

subcluster_info = []
for cluster_id in range(n_clusters):

    cluster_indices = np.where(fcm_labels == cluster_id)[0]
    cluster_points = points_3d[cluster_indices]
    
    if len(cluster_points) < 10:
        continue
    
    print(f" {cluster_id}: {len(cluster_points)} points")
    
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=5,
        min_samples=3,
        cluster_selection_epsilon=0.05,
        metric='euclidean'
    )
    
    hdbscan_labels = clusterer.fit_predict(cluster_points)
    
    unique_hdbscan_labels = np.unique(hdbscan_labels)
    n_subclusters = len(unique_hdbscan_labels) - (1 if -1 in unique_hdbscan_labels else 0)

    
    for hdb_label in unique_hdbscan_labels:
        if hdb_label != -1:
            subcluster_indices_in_cluster = np.where(hdbscan_labels == hdb_label)[0]
            subcluster_size = len(subcluster_indices_in_cluster)
            subcluster_info.append({
                'fcm_cluster': cluster_id,
                'hdbscan_label': hdb_label,
                'global_label': subcluster_counter,
                'indices': cluster_indices[subcluster_indices_in_cluster],
                'size': subcluster_size
            })
            subcluster_counter += 1

for i, info in enumerate(subcluster_info):

    color_idx = i % len(distinct_colors)
    color = distinct_colors[color_idx]
    
    all_hdbscan_labels[info['indices']] = info['global_label']
    final_colors[info['indices']] = color
    
    print(f" {info['global_label']:2d} (FCM{info['fcm_cluster']}): {info['size']:5d} points, RGB: [{color[0]:.2f}, {color[1]:.2f}, {color[2]:.2f}]")

noise_indices = np.where(all_hdbscan_labels == -1)[0]
if len(noise_indices) > 0:
    final_colors[noise_indices] = [0.1, 0.1, 0.1] 



total_points = len(fcm_labels)
valid_subclusters = len(subcluster_info)
noise_points = len(noise_indices)

total_time = time.time() - start
print(f"{total_time:.2f} ")

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points_3d)
pcd.colors = o3d.utility.Vector3dVector(final_colors)
o3d.visualization.draw_geometries([pcd], window_name="FCM+HDBSCAN")

In [52]:
header = laspy.LasHeader(point_format=3, version="1.3")
header.offsets = np.min(lasdata, axis=0)

header.scales = [0.001, 0.001, 0.001]
las = laspy.LasData(header)

las.x = points_3d[:, 0]
las.y = points_3d[:, 1]
las.z = points_3d[:, 2]

las.red = final_colors[:, 0] * 65535
las.green = final_colors[:, 1] * 65535
las.blue = final_colors[:, 2] * 65535

output_file = r"XXX.las"
las.write(output_file)

# Normals neighborhood deviation+DPC

In [19]:
def compute_curvature_based_on_normal_difference(points, normals, k=30):
    num_points = len(points)

    kdtree = KDTree(points)

    curvatures = np.zeros(num_points)

    distances, indices = kdtree.query(points, k=k+1)  # distances: (n, k+1), indices: (n, k+1)

    for i in range(num_points):
        normal_i = normals[i]
        neighbor_indices = indices[i]

        neighbor_indices = neighbor_indices[neighbor_indices != i]

        if len(neighbor_indices) == 0:
            curvature = 0.0
        else:
            neighbor_normals = normals[neighbor_indices]
            dot_products = np.clip(np.dot(neighbor_normals, normal_i), -1.0, 1.0)
            angle_diffs = np.arccos(dot_products)
            curvature = np.mean(angle_diffs)
        curvatures[i] = curvature
    return curvatures

In [20]:
curvatures = compute_curvature_based_on_normal_difference(lasdata, nls)

In [21]:
curvatures = np.array(curvatures)
curvatures = curvatures.reshape(-1,1)
len(curvatures)

1044090

In [54]:
start = time.time()

In [55]:
X = curvatures 
K = 6
kmeans = KMeans(n_clusters=K, init='k-means++', random_state=0).fit(X)
labels_k = kmeans.labels_
centers = kmeans.cluster_centers_
cluster_k = [lasdata[labels_k == i] for i in range(K)]

In [56]:
sorted_clusters = sorted(cluster_k, key=lambda x: len(x), reverse=True)
clusters_with_indices = [(i, cluster_k[i]) for i in range(K)]
sorted_clusters_with_indices = sorted(clusters_with_indices, key=lambda x: len(x[1]), reverse=True)
sorted_indices = [item[0] for item in sorted_clusters_with_indices]
sorted_arrays = [item[1] for item in sorted_clusters_with_indices]

In [57]:
points0 = sorted_arrays[0]  
index0 = find_point_indices(points0, lasdata)
normals0 = nls[index0]

In [58]:
qxqj = []
for i in range(len(normals0)):
    qxqj.append((arcsin_and_arccos(normals0[i]),arctans(normals0[i])))

In [59]:
qxqj = np.array(qxqj)

In [12]:
full_labels, sampled_labels, sample_indices = cfsfdp_sampling(
    qxqj, 
    sample_ratio=0.001,
    n_neighbors=50,
    n_centers=5)
print(f"{len(np.unique(full_labels))}clusters ")
print(f"{np.bincount(full_labels)}")

In [63]:
manual_colors = {
    0: [113,184,158],  
    1: [246,136,239],   
    2: [33,106,212],
    3: [184,71,102], 
    4: [254,234,114]   
}

cluster_colors = {}
for cluster_id in set(full_labels):
    if cluster_id == -1:
        cluster_colors[cluster_id] = [0.5, 0.5, 0.5]
    elif cluster_id in manual_colors:

        color_rgb = manual_colors[cluster_id]
        cluster_colors[cluster_id] = [c / 255.0 for c in color_rgb]
    else:

        cluster_colors[cluster_id] = [random.random(), random.random(), random.random()]

colors = []
for label in full_labels:
    colors.append(cluster_colors[label])

colors = np.array(colors)

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points0)
pcd.colors = o3d.utility.Vector3dVector(colors)

o3d.visualization.draw_geometries([pcd], window_name="result")

In [14]:
unique_labels = np.unique(full_labels)
stats = {}
for label in unique_labels:
    if label == -1:
        continue  
    mask = (full_labels == label)
    cluster_points = qxqj[mask]
    centroid = np.mean(cluster_points, axis=0)  
    print(centroid)

In [13]:
combined_pcd = simple_process_all_clusters_tab20(points0, full_labels)
total_time = time.time() - start
print(f"{total_time:.2f}")

In [32]:
header = laspy.LasHeader(point_format=3, version="1.3")
header.offsets = np.min(sorted_arrays[0], axis=0)

header.scales = [0.001, 0.001, 0.001]
las = laspy.LasData(header)

las.x = points0[:, 0]
las.y = points0[:, 1]
las.z = points0[:, 2]

las.red = colors[:, 0] * 65535
las.green = colors[:, 1] * 65535
las.blue = colors[:, 2] * 65535

output_file = r"XXX.las"
las.write(output_file)